In [0]:
%pip install torch torch-geometric
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import torch

from torch_geometric.data import Data
from sklearn.preprocessing import LabelEncoder

In [0]:
clientes = spark.table("clientes_features")
edges = spark.table("edges_raw")

In [0]:
pdf_clientes = clientes.toPandas()
pdf_edges = edges.toPandas()

In [0]:
features = [
    "idade",
    "renda_mensal",
    "score_credito",
    "qtd_produtos",
    "valor_fraude",
    "tempo_resolucao_dias",
    "stress_cliente",
    "pix_por_renda"
]

X = pdf_clientes[features].fillna(0).values

In [0]:
y = pdf_clientes["judicializou"].values

In [0]:
# Todos os nós existentes nas arestas
all_nodes = pd.concat([
    pdf_edges["source"],
    pdf_edges["target"]
]).unique()

node_map = {node: i for i, node in enumerate(all_nodes)}

num_nodes = len(all_nodes)
num_features = len(features)

# matriz zerada para todos os nós
X_all = np.zeros((num_nodes, num_features), dtype=np.float32)

# target padrão -1 para nós que não são clientes
y_all = np.full(num_nodes, -1, dtype=np.int64)

In [0]:
for _, row in pdf_clientes.iterrows():
    node_name = f"cliente_{int(row['cliente_id'])}"

    if node_name in node_map:
        idx = node_map[node_name]

        X_all[idx] = row[features].fillna(0).values.astype(np.float32)
        y_all[idx] = int(row["judicializou"])

In [0]:
pdf_edges["source_id"] = pdf_edges["source"].map(node_map)
pdf_edges["target_id"] = pdf_edges["target"].map(node_map)

edge_index = torch.tensor(
    pdf_edges[["source_id","target_id"]].values.T,
    dtype=torch.long
)

In [0]:
x = torch.tensor(X_all, dtype=torch.float)
y = torch.tensor(y_all, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

print(data)
print("Nós:", data.num_nodes)
print("Features:", data.num_features)
print("Maior índice edge:", int(data.edge_index.max()))

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from torch_geometric.nn import GCNConv, SAGEConv
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [0]:
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [0]:
cliente_mask = data.y != -1
cliente_idx = cliente_mask.nonzero(as_tuple=False).view(-1)

idx = cliente_idx[torch.randperm(cliente_idx.size(0))]

train_size = int(0.7 * len(idx))
val_size = int(0.15 * len(idx))

train_idx = idx[:train_size]
val_idx = idx[train_size:train_size + val_size]
test_idx = idx[train_size + val_size:]

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

data.train_mask[train_idx] = True
data.val_mask[val_idx] = True
data.test_mask[test_idx] = True

In [0]:
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        embeddings = x

        x = self.conv2(x, edge_index)
        return x, embeddings

In [0]:
model = GCN(
    in_channels=data.x.shape[1],
    hidden_channels=32,
    out_channels=2
).to(device)

data = data.to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

loss_fn = nn.CrossEntropyLoss()

In [0]:
def train():
    model.train()
    optimizer.zero_grad()

    out, embeddings = model(data.x, data.edge_index)

    loss = loss_fn(out[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()

    return loss.item()

In [0]:
@torch.no_grad()
def evaluate(mask):
    model.eval()

    out, embeddings = model(data.x, data.edge_index)

    probs = torch.softmax(out[mask], dim=1)[:, 1].cpu().numpy()
    preds = out[mask].argmax(dim=1).cpu().numpy()
    y_true = data.y[mask].cpu().numpy()

    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "auc": roc_auc_score(y_true, probs)
    }

In [0]:
for epoch in range(1, 101):
    loss = train()

    if epoch % 10 == 0:
        val_metrics = evaluate(data.val_mask)
        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Val AUC: {val_metrics['auc']:.4f} | "
            f"Val Recall: {val_metrics['recall']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f}"
        )

In [0]:
test_metrics = evaluate(data.test_mask)
test_metrics

### Visualizacao

In [0]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from torch_geometric.utils import to_networkx

In [0]:
model.eval()

with torch.no_grad():
    out, embeddings = model(data.x, data.edge_index)

emb = embeddings.cpu().numpy()

In [0]:
cliente_mask = (data.y != -1).cpu().numpy()

emb_clientes = emb[cliente_mask]
y_clientes = data.y.cpu().numpy()[cliente_mask]

In [0]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)

clusters = kmeans.fit_predict(emb_clientes)

In [0]:
tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30,
    learning_rate="auto",
    init="pca"
)

emb_2d = tsne.fit_transform(emb_clientes)

In [0]:
plt.figure(figsize=(10, 7))

plt.scatter(
    emb_2d[:, 0],
    emb_2d[:, 1],
    c=clusters,
    cmap="tab10",
    s=8,
    alpha=0.7
)

plt.title("Personas detectadas pela GNN")
plt.xlabel("Dimensão 1")
plt.ylabel("Dimensão 2")
plt.colorbar(label="Cluster / Persona")
plt.show()

In [0]:
plt.figure(figsize=(10, 7))

plt.scatter(
    emb_2d[:, 0],
    emb_2d[:, 1],
    c=clusters,
    cmap="tab10",
    s=8,
    alpha=0.7
)

plt.title("Personas detectadas pela GNN")
plt.xlabel("Dimensão 1")
plt.ylabel("Dimensão 2")
plt.colorbar(label="Cluster / Persona")
plt.show()

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from torch_geometric.utils import to_networkx

# Grafo completo
G = to_networkx(data.cpu(), to_undirected=True)

# índices dos nós que são clientes
cliente_node_ids = np.where((data.y.cpu().numpy() != -1))[0]

# mapa: nó cliente -> cluster/persona
node_to_cluster = {
    int(node_id): int(cluster)
    for node_id, cluster in zip(cliente_node_ids, clusters)
}

# amostra de clientes
sample_size = min(800, len(cliente_node_ids))
sample_clientes = np.random.choice(cliente_node_ids, size=sample_size, replace=False)

# incluir também vizinhos dos clientes, para aparecerem conexões
nodes_to_keep = set(sample_clientes)

for node in sample_clientes:
    nodes_to_keep.update(G.neighbors(int(node)))

subG = G.subgraph(nodes_to_keep).copy()

# cores:
# cliente recebe cluster
# nó auxiliar recebe cinza/classe padrão -1
node_colors = []
node_sizes = []

for node in subG.nodes():
    if node in node_to_cluster:
        node_colors.append(node_to_cluster[node])
        node_sizes.append(25)
    else:
        node_colors.append(-1)
        node_sizes.append(8)

plt.figure(figsize=(14, 10))

pos = nx.spring_layout(subG, seed=42, k=0.12)

nx.draw_networkx_edges(
    subG,
    pos,
    alpha=0.08,
    width=0.5
)

nx.draw_networkx_nodes(
    subG,
    pos,
    node_color=node_colors,
    cmap=plt.cm.tab10,
    node_size=node_sizes,
    alpha=0.85
)

plt.title("Rede de comunidades / personas detectadas pela GNN")
plt.axis("off")
plt.show()

In [0]:
pdf_clientes = spark.table("clientes_features").toPandas()
pdf_clientes["persona_gnn"] = clusters

In [0]:
resumo_personas = (
    pdf_clientes
    .groupby("persona_gnn")
    .agg({
        "cliente_id": "count",
        "judicializou": "mean",
        "idade": "mean",
        "renda_mensal": "mean",
        "valor_fraude": "mean",
        "tempo_resolucao_dias": "mean",
        "stress_cliente": "mean",
        "pix_por_renda": "mean"
    })
    .reset_index()
)

resumo_personas["taxa_judicializacao"] = resumo_personas["judicializou"] * 100

display(resumo_personas)

In [0]:
def moda(s):
    return s.mode().iloc[0] if not s.mode().empty else None

perfil_categorico = (
    pdf_clientes
    .groupby("persona_gnn")
    .agg(
        segmento_dominante=("segmento", moda),
        tipo_fraude_dominante=("tipo_fraude", moda),
        canal_dominante=("canal_preferencial", moda),
        produto_dominante=("produto_principal", moda),
        escolaridade_dominante=("escolaridade", moda)
    )
    .reset_index()
)

display(perfil_categorico)

In [0]:
resumo_final_personas = resumo_personas.merge(
    perfil_categorico,
    on="persona_gnn",
    how="left"
)

display(resumo_final_personas)

In [0]:
pdf_clientes["persona_gnn"] = clusters

In [0]:
clientes_features_persona = spark.createDataFrame(pdf_clientes)

In [0]:
clientes_features_persona.write.mode("overwrite").saveAsTable("clientes_features_persona")